### 说明
用于演示 AutoGen 框架下的多代理协作系统，具体展示了如何在旅行规划场景中实现两个具有不同角色的AI代理之间的协作。
1. **多代理系统架构**
    - 创建了两个具有明确角色分工的Agent：
        - `frontdesk_agent`（前台旅行代理）：专注于提供简洁、高效的旅行推荐
        - `concierge_agent`（酒店礼宾）：评估推荐质量并提供本地化改进建议
2. **业务场景实现**
    - 模拟了用户请求"我想计划一次巴黎之旅"的完整对话流程
    - 通过轮询式对话机制(`RoundRobinGroupChat`)实现代理间的有序交互
    - 设置了明确的终止条件(`TextMentionTermination`)：当礼宾回复"APPROVE"时结束对话
3. **框架特性展示**
    - 展示了AutoGen的流式响应处理(`run_stream`)
    - 演示了Azure AI服务集成配置
    - 体现了多代理协作的工作流程和消息传递机制

In [3]:
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient 
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken
from autogen_agentchat.base import TaskResult

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# 使用通义大模型初始化 OpenAI 兼容的模型客户端
client = OpenAIChatCompletionClient(
    model="qwen-max", # 指定使用的模型ID为通义千问的qwen-max。
    # 已修改：更换为 DashScope 兼容模式的 API 基础 URL
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
    # 已修改：对于兼容 OpenAI API 的服务，直接将 Key 传入 api_key 参数
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 提供模型的额外信息，指导 AutoGen 如何与模型交互
    model_info={
        "json_output": True, # 表明模型支持 JSON 格式输出。
        "structured_output": True, # 表明模型支持结构化输出。
        "function_calling": True, # 表明模型支持工具调用/Function Calling。
        "vision": True, # 表明模型支持视觉（多模态）能力。
        "family": "unknown", # 模型家族信息（这里设置为未知）。
    },
)

In [2]:
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
    # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\autogen_ext\models\azure\_azure_ai_client.py:307: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(config["model_info"])


In [6]:
# 创建一个名为 frontdesk_agent 的 AssistantAgent 实例对象
frontdesk_agent = AssistantAgent(
    # 代理的唯一标识名称，通常用于系统内部调度或日志记录
    "planner_agent",
    # 指定该代理使用的模型客户端（如配置好的 OpenAI 或其他 LLM 客户端）
    model_client=client,
    # 代理的功能描述：这有助于编排者（Orchestrator）判断何时该调用这个特定的代理
    description="A helpful assistant that can plan trips.",
    # 系统消息（人设指令）：定义了 Agent 的角色、性格、行为准则和输出约束
    system_message="""
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.""",
)


# 您是一位拥有十年经验的前台旅行社代理，在与众多客户打交道时以简洁著称。
# 目标是为旅行者提供最佳的活动和游览地点。
# 每个回复仅提供一条建议。
# 您全神贯注于当前的目标。
# 不要浪费时间闲聊。
# 在完善想法时考虑建议。


# 创建名为 concierge_agent 的助手代理实例
concierge_agent = AssistantAgent(
    # 代理的唯一名称标识，系统调度时会用到
    "concierge_agent",
    # 连接到大模型的客户端（如 OpenAI 或本地 LLM）
    model_client=client,
    # 代理的功能描述：告诉编排系统，这是一个能提供当地活动和地点建议的本地助手
    description="A local assistant that can suggest local activities or places to visit.",
    # 系统消息（核心逻辑）：定义了代理的行为边界和评判标准
    system_message="""
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, respond with 'APPROVE'
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """,
)

# 您是一位酒店礼宾员，对如何为旅客提供最本地化、最真实的体验有着自己的看法。
# 目标是确定前台旅行社是否为旅行者推荐了最佳的非旅游体验。
# 如果是，请回复“批准”
# 如果没有，请提供如何在不使用具体示例的情况下改进建议的见解。


In [7]:
# --- 1. 定义终止条件 ---
# 实例化一个终止对象：当对话中出现 "APPROVE" 这个词时，系统会自动停止运行。
# 这通常用于由“审计者”代理发出的信号，表示建议已经达标。
termination = TextMentionTermination("APPROVE")
# --- 2. 组建团队与运行规则 ---
# 创建一个采用“轮询模式（Round Robin）”的群聊团队。
# [frontdesk_agent, concierge_agent]: 设定代理发言顺序，A 说完 B 说，以此往复。
# termination_condition: 将前面定义的终止条件绑定到团队中。
team = RoundRobinGroupChat(
    [frontdesk_agent, concierge_agent], termination_condition=termination
)
# --- 3. 启动异步流式任务 ---
# 使用异步迭代器运行团队任务。task 是交给团队的初始指令。
# run_stream 会实时返回代理之间的对话片段（Streaming），而不是等全部运行完才一次性返回。
async for message in team.run_stream(task="I would like to plan a trip to Paris."): 
    # 检查当前消息是否为“任务最终结果对象”
    if isinstance(message, TaskResult):
        # 如果是结果对象，打印停止的原因（例如：是因为看到了 "APPROVE" 而停止的）
        print("Stop Reason:", message.stop_reason)
    else:
        # 如果是普通的代理对话内容，直接将其打印出来，方便我们实时观察 Agent 之间的交流
        print(message)

id='55f88c73-f106-493f-828f-4756f5338c30' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 4, 26, 15, 26, 1, 334632, tzinfo=datetime.timezone.utc) content='I would like to plan a trip to Paris.' type='TextMessage'
id='99ccad6d-1498-4419-b70f-bba34cfc5728' source='planner_agent' models_usage=RequestUsage(prompt_tokens=105, completion_tokens=13) metadata={} created_at=datetime.datetime(2026, 4, 26, 15, 26, 2, 557238, tzinfo=datetime.timezone.utc) content='Start with the Eiffel Tower for iconic views and photos.' type='TextMessage'
id='7897a631-7853-434e-b14b-2bb351c30995' source='concierge_agent' models_usage=RequestUsage(prompt_tokens=112, completion_tokens=98) metadata={} created_at=datetime.datetime(2026, 4, 26, 15, 26, 7, 851747, tzinfo=datetime.timezone.utc) content="While the Eiffel Tower is undeniably an iconic symbol of Paris and a must-see for many, it's also one of the most touristy spots in the city. To provide a more local and authentic experienc


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
